# Distributed estimation on Spark

mixle's estimation functions (`initialize`, `optimize`, `seq_encode`, `seq_estimate`, ...) accept either in-memory lists or Spark RDDs - the code is identical, only the data argument changes. Under the hood each partition encodes its rows, runs the E-step locally, and ships back sufficient statistics, which are merged on the driver via the accumulator `combine` protocol before the M-step. Sufficient statistics are tiny compared to data, so the only per-iteration traffic is one small object per partition.

This notebook fits two models distributively: an HMM mixture on sentences of the Iliad, and a composite mixture on data sampled in parallel on the cluster (the distribution itself is shipped to workers along with seeds).

Requirements: `pip install git+https://github.com/gmboquet/mixle.git`, plus a JVM for Spark - PySpark 4.x needs Java 17 or 21 (e.g. `brew install openjdk@17` and set `JAVA_HOME`).


In [1]:
import sys
import os
# Workers must run the same Python as the driver.
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable
# If Java is not on PATH, point JAVA_HOME at a JDK 17/21 install, e.g.:
# os.environ['JAVA_HOME'] = '/opt/homebrew/opt/openjdk@17/libexec/openjdk.jdk/Contents/Home'

Running `local[*]` (this notebook), workers share the driver's Python environment, so installing mixle is all that's needed. On a real cluster, install mixle on the workers (or pass a wheel via `SparkContext(..., pyFiles=[...])` / `spark-submit --py-files`).


In [2]:
import re
import pandas as pd
import numpy as np
import itertools
import matplotlib.pyplot as plt
from bokeh.io import output_notebook, show
from bokeh.plotting import figure
from sklearn.manifold import TSNE
from pyspark import SparkContext, SparkConf


from mixle.stats import (
    BernoulliSetDistribution, BernoulliSetEstimator, CategoricalDistribution,
    CategoricalEstimator, CompositeDistribution, CompositeEstimator, GaussianDistribution,
    GaussianEstimator, HiddenMarkovEstimator, MarkovChainDistribution, MarkovChainEstimator,
    MixtureDistribution, MixtureEstimator, MultivariateGaussianDistribution,
    MultivariateGaussianEstimator, OptionalDistribution, OptionalEstimator,
    PoissonDistribution, PoissonEstimator, seq_encode)
from mixle.data import sample_rdd
from mixle.inference.estimation import optimize
from mixle.utils.optsutil import map_to_integers, get_inv_map

output_notebook()

Loading BokehJS ...

In [3]:
conf = SparkConf().setAppName('mixle_example')
sc = SparkContext(conf=conf)
sc.setLogLevel('ERROR')

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties


26/07/11 19:14:28 WARN Utils: Your hostname, GMB.local, resolves to a loopback address: 127.0.0.1; using 192.168.21.41 instead (on interface en0)
26/07/11 19:14:28 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/07/11 19:14:46 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## Example with existing data


In [4]:
# Load the Iliad and split it into sentence-like chunks
with open('../../data/iliad/iliad_en.txt', encoding='utf-8') as fin:
    text = fin.read()

# drop the Gutenberg header/footer, then split on sentence punctuation
body = text[text.find('BOOK I.'):text.rfind('END OF THE PROJECT')]
sentences = re.split(r'[.;?!]+', body)
iliad = [re.split(r'\s+', s.strip()) for s in sentences]
iliad = [u for u in iliad if len(u) > 3]
iliad = iliad[:200]

# Make the RDD
iliad_rdd = sc.parallelize(iliad)

The model: a 10-component mixture of 10-state HMMs over words, with the HMM emission statistics tied across mixture components through the `keys` mechanism (accumulators sharing a key merge their sufficient statistics).


In [5]:
est1 = CategoricalEstimator(pseudo_count=1.0)
est2 = HiddenMarkovEstimator([est1]*10, keys=(None, None, 'comp'), pseudo_count=(1.0,1.0), len_estimator=PoissonEstimator())
est  = MixtureEstimator([est2]*10)

Distributed model parameter estimation looks exactly the same as the local version. PySparkPlug uses type checking of the passed data to determine the appropriate response.


In [6]:
model = optimize(iliad_rdd, est, max_its=5, rng=np.random.RandomState(1))

Iteration 1: ln[p_mat(Data|Model)]=-2.514120e+04, ln[p_mat(Data|Model)]-ln[p_mat(Data|PrevModel)]=5.195569e+03


Iteration 2: ln[p_mat(Data|Model)]=-2.476606e+04, ln[p_mat(Data|Model)]-ln[p_mat(Data|PrevModel)]=3.751356e+02


Iteration 3: ln[p_mat(Data|Model)]=-2.442212e+04, ln[p_mat(Data|Model)]-ln[p_mat(Data|PrevModel)]=3.439450e+02


Iteration 4: ln[p_mat(Data|Model)]=-2.409838e+04, ln[p_mat(Data|Model)]-ln[p_mat(Data|PrevModel)]=3.237429e+02


Iteration 5: ln[p_mat(Data|Model)]=-2.382605e+04, ln[p_mat(Data|Model)]-ln[p_mat(Data|PrevModel)]=2.723221e+02


The overhead of a distributed computation means it is not always faster.


In [7]:
model = optimize(iliad, est, max_its=100, prev_estimate=model, print_iter=10)

Iteration 10: ln[p_mat(Data|Model)]=-2.295521e+04, ln[p_mat(Data|Model)]-ln[p_mat(Data|PrevModel)]=3.782819e+01


Iteration 20: ln[p_mat(Data|Model)]=-2.278820e+04, ln[p_mat(Data|Model)]-ln[p_mat(Data|PrevModel)]=9.942677e+00


Iteration 30: ln[p_mat(Data|Model)]=-2.272507e+04, ln[p_mat(Data|Model)]-ln[p_mat(Data|PrevModel)]=3.109506e+00


Iteration 40: ln[p_mat(Data|Model)]=-2.269182e+04, ln[p_mat(Data|Model)]-ln[p_mat(Data|PrevModel)]=2.803008e+00


Iteration 50: ln[p_mat(Data|Model)]=-2.266260e+04, ln[p_mat(Data|Model)]-ln[p_mat(Data|PrevModel)]=1.575289e+00


Iteration 60: ln[p_mat(Data|Model)]=-2.265078e+04, ln[p_mat(Data|Model)]-ln[p_mat(Data|PrevModel)]=1.446761e+00


Iteration 70: ln[p_mat(Data|Model)]=-2.263597e+04, ln[p_mat(Data|Model)]-ln[p_mat(Data|PrevModel)]=8.348528e-01


Iteration 80: ln[p_mat(Data|Model)]=-2.262187e+04, ln[p_mat(Data|Model)]-ln[p_mat(Data|PrevModel)]=9.105692e-01


Iteration 90: ln[p_mat(Data|Model)]=-2.260662e+04, ln[p_mat(Data|Model)]-ln[p_mat(Data|PrevModel)]=9.179207e-01


Iteration 100: ln[p_mat(Data|Model)]=-2.259154e+04, ln[p_mat(Data|Model)]-ln[p_mat(Data|PrevModel)]=1.459600e+00


Compute the mixture posterior probabilties and create a dataframe.


In [8]:
posteriors = model.seq_posterior(model.dist_to_encoder().seq_encode(iliad))
coords = TSNE(random_state=1).fit_transform(posteriors)

iliad_df = pd.DataFrame([{'x': coords[i,0], 'y': coords[i,1], 'text': ' '.join(iliad[i])} for i in range(len(iliad))])

And plot the the results using the Bokeh scatter plot.


In [9]:
p = figure(title = "Iliad Embedded Sentences", tooltips = """<div style="width:200px;">@text</div>""")
p.xaxis.axis_label = 't-SNE X'
p.yaxis.axis_label = 't-SNE Y'
p.scatter("x", "y", source=iliad_df)

show(p)

## Example with sampled data


We'll start off with random data that we know. Random sampling is performed by distributing to each node the distribution and a random seed.


In [10]:
# -- Specify data distribution --- 

d10 = MixtureDistribution([GaussianDistribution(-3.0, 1.0), GaussianDistribution(0.0, 1.0)], [0.5, 0.5])
d11 = OptionalDistribution(CategoricalDistribution({'a': 0.5, 'b': 0.4, 'c': 0.1}), p=0.1)
d12 = MarkovChainDistribution({'a' : 0.5, 'b' : 0.5}, {'a' : { 'a' : 0.2, 'b' : 0.8}, 'b' : { 'a' : 0.8, 'b' : 0.2}}, len_dist=PoissonDistribution(8.0))
d13 = BernoulliSetDistribution({'a' : 0.1, 'b': 0.3})
d14 = MultivariateGaussianDistribution([-1.0, -1.0], [[2.0, 1.0], [1.0, 2.0]])
d1  = CompositeDistribution([d10, d11, d12, d13, d14])

d20 = MixtureDistribution([GaussianDistribution(0.0, 1.0), GaussianDistribution(6.0, 1.0)], [0.5, 0.5])
d21 = OptionalDistribution(CategoricalDistribution({'a': 0.1, 'b': 0.1, 'c': 0.8}), p=0.2)
d22 = MarkovChainDistribution({'a': 0.5, 'b': 0.5}, {'a': {'a': 0.8, 'b': 0.2}, 'b': {'a': 0.2, 'b': 0.8}}, len_dist=PoissonDistribution(8.0))
d23 = BernoulliSetDistribution({'a': 0.9, 'b': 0.8})
d24 = MultivariateGaussianDistribution([1.0, 1.0], [[2.0, 1.0], [1.0, 2.0]])
d2  = CompositeDistribution([d20, d21, d22, d23, d24])

dist = MixtureDistribution([d1, d2], [0.5, 0.5])


# -- Sample data --- 

rng = np.random.RandomState(2)

# sample_rdd is an RDD
data = sample_rdd(sc, dist, count_per_split=200, num_splits=10, seed=rng.randint(2**31))

# split into train and validation data 
train_data, valid_data = data.randomSplit([0.9, 0.1], seed=rng.randint(2**31))

Take an entry from the RDD


In [11]:
train_data.take(1)

[(np.float64(-4.327826169894319),
  'a',
  ['b', 'a', 'b', 'a', 'a', 'b', 'a', 'b', 'a'],
  [],
  array([-0.87117281,  0.5452792 ]))]

Create a mixle estimator for the data.


In [12]:
e0 = MixtureEstimator([GaussianEstimator()] * 2)
e1 = OptionalEstimator(CategoricalEstimator(pseudo_count=1.0), est_prob=True, pseudo_count=1.0)
e2 = MarkovChainEstimator(pseudo_count=1.0, len_estimator=PoissonEstimator())
e3 = BernoulliSetEstimator()
e4 = MultivariateGaussianEstimator()
est = MixtureEstimator([CompositeEstimator((e0, e1, e2, e3, e4))] * 2)

Run some iterations of EM


In [13]:
mm = optimize(train_data, est, max_its=10, rng=np.random.RandomState(1))

Iteration 1: ln[p_mat(Data|Model)]=-2.963041e+04, ln[p_mat(Data|Model)]-ln[p_mat(Data|PrevModel)]=9.224517e+02


Iteration 2: ln[p_mat(Data|Model)]=-2.707313e+04, ln[p_mat(Data|Model)]-ln[p_mat(Data|PrevModel)]=2.557289e+03


Iteration 3: ln[p_mat(Data|Model)]=-2.685082e+04, ln[p_mat(Data|Model)]-ln[p_mat(Data|PrevModel)]=2.223066e+02


Iteration 4: ln[p_mat(Data|Model)]=-2.684953e+04, ln[p_mat(Data|Model)]-ln[p_mat(Data|PrevModel)]=1.292950e+00


Iteration 5: ln[p_mat(Data|Model)]=-2.684931e+04, ln[p_mat(Data|Model)]-ln[p_mat(Data|PrevModel)]=2.146689e-01


Iteration 6: ln[p_mat(Data|Model)]=-2.684926e+04, ln[p_mat(Data|Model)]-ln[p_mat(Data|PrevModel)]=4.952421e-02


Iteration 7: ln[p_mat(Data|Model)]=-2.684925e+04, ln[p_mat(Data|Model)]-ln[p_mat(Data|PrevModel)]=1.187994e-02


Iteration 8: ln[p_mat(Data|Model)]=-2.684925e+04, ln[p_mat(Data|Model)]-ln[p_mat(Data|PrevModel)]=2.891877e-03


Iteration 9: ln[p_mat(Data|Model)]=-2.684925e+04, ln[p_mat(Data|Model)]-ln[p_mat(Data|PrevModel)]=7.253311e-04


Iteration 10: ln[p_mat(Data|Model)]=-2.684925e+04, ln[p_mat(Data|Model)]-ln[p_mat(Data|PrevModel)]=2.019126e-04


Run some more iterations


In [14]:
mm = optimize(train_data, est, max_its=10, prev_estimate=mm)

Iteration 1: ln[p_mat(Data|Model)]=-2.684925e+04, ln[p_mat(Data|Model)]-ln[p_mat(Data|PrevModel)]=7.563197e-05


Iteration 2: ln[p_mat(Data|Model)]=-2.684925e+04, ln[p_mat(Data|Model)]-ln[p_mat(Data|PrevModel)]=4.545864e-05


Iteration 3: ln[p_mat(Data|Model)]=-2.684925e+04, ln[p_mat(Data|Model)]-ln[p_mat(Data|PrevModel)]=3.855585e-05


Iteration 4: ln[p_mat(Data|Model)]=-2.684925e+04, ln[p_mat(Data|Model)]-ln[p_mat(Data|PrevModel)]=3.729148e-05


Iteration 5: ln[p_mat(Data|Model)]=-2.684925e+04, ln[p_mat(Data|Model)]-ln[p_mat(Data|PrevModel)]=3.739774e-05


Iteration 6: ln[p_mat(Data|Model)]=-2.684925e+04, ln[p_mat(Data|Model)]-ln[p_mat(Data|PrevModel)]=3.784247e-05


Iteration 7: ln[p_mat(Data|Model)]=-2.684925e+04, ln[p_mat(Data|Model)]-ln[p_mat(Data|PrevModel)]=3.837662e-05


Iteration 8: ln[p_mat(Data|Model)]=-2.684925e+04, ln[p_mat(Data|Model)]-ln[p_mat(Data|PrevModel)]=3.894043e-05


Iteration 9: ln[p_mat(Data|Model)]=-2.684925e+04, ln[p_mat(Data|Model)]-ln[p_mat(Data|PrevModel)]=3.951983e-05


Iteration 10: ln[p_mat(Data|Model)]=-2.684925e+04, ln[p_mat(Data|Model)]-ln[p_mat(Data|PrevModel)]=4.011176e-05


In [15]:
print(str(mm))

MixtureDistribution([CompositeDistribution((MixtureDistribution([GaussianDistribution(-1.6401845726928717, 3.2500922116110713, name=None),GaussianDistribution(-1.3398468042944072, 3.235310638625349, name=None)], [np.float64(0.5785545952892431), np.float64(0.4214454047107568)], name=None),OptionalDistribution(CategoricalDistribution({'a': np.float64(0.5131646035994833), 'b': np.float64(0.3882588453122816), 'c': np.float64(0.09857655108823499)}, default_value=0.0, name=None), p=np.float64(0.11383744504503095), missing_value=None, name=None),MarkovChainDistribution({np.str_('a'): np.float64(0.5301602077486669), np.str_('b'): np.float64(0.46983979225133304)}, {np.str_('a'): {np.str_('a'): np.float64(0.21042158922767645), np.str_('b'): np.float64(0.7895784107723236)}, np.str_('b'): {np.str_('a'): np.float64(0.798061251256313), np.str_('b'): np.float64(0.2019387487436869)}}, len_dist=PoissonDistribution(8.05339847464812, name=None), default_value=np.float64(0.00013322374949096934), name=None